In [ ]:
import enrich_omics
import os
from enrich_omics import EnrichR
from enrich_omics import OpenTargets
import shutil

# get all available libraries
EnrichR.get_libraries()
librerie=['OMIM_Disease','DisGeNET','KEGG_2021_Human','GO_Biological_Process_2025','GO_Cellular_Component_2025','GO_Molecular_Function_2025']

In [5]:
print(enrich_omics.__file__)

/home/maria/anaconda3/lib/python3.9/site-packages/enrich_omics/__init__.py


In [ ]:
#EnrichR.get_libraries()
import itertools


In [ ]:
#pip install vl-convert-python
import altair as alt
alt.renderers.enable("png", scale_factor=2)   # 2x resolution


In [ ]:
import pandas as pd

In [ ]:
comparisons=['bAmy-vs-CTR','CTR-Tau-vs-CTR','PCB118-vs-CTR','PCB153-vs-CTR']

In [ ]:
import matplotlib.pyplot as plt

def plot_combined_score(df, save_path=None, dpi=300, show_plot=True):
    """
    Plots the Combined Score for enriched terms in descending order.
    
    Parameters:
    - df: pandas DataFrame containing at least 'Term name' and 'Combined score'.
    - save_path: str, optional path to save the figure.
    - dpi: int, resolution of saved figure.
    - show_plot: bool, whether to display the plot.
    """
    # sort by Combined score
    df_sorted = df.sort_values('Adjusted p-value',ascending=False)
    
    plt.figure(figsize=(10, max(4, len(df_sorted) * 0.4)))
    plt.barh(df_sorted['Term name'], df_sorted['number of Genes'], color='skyblue')
    plt.xlabel('number of Genes')
    plt.title('Enrichment ordered by Adjusted p-value')
    plt.tight_layout()
    
    # save the figure if save_path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi)
    
    # show the plot if requested
    if show_plot:
        plt.show()
    else:
        plt.close()


In [ ]:
D_comparison={}
for comp in comparisons:
    DASE_df=pd.read_csv('%s/DASE/de_novo_annotated_DASE_%s.txt'%(comp,comp),sep='\t')
    dase_genes=set(DASE_df['geneSymbol'].dropna())
    D_comparison[comp]=dase_genes



In [ ]:
D_shared = {}

for r in range(2, len(comparisons) + 1):
    print(f"\nCombinations of {r}:")
    for combo in itertools.combinations(comparisons, r):

        # Compute shared genes for ANY combo size
        shared_genes = D_comparison[combo[0]].copy()
        for comp in combo[1:]:
            shared_genes = shared_genes.intersection(D_comparison[comp])

        # Key = directory name + prefix for file
        key = "_AND_".join(combo)
        D_shared[key] = shared_genes

        # ------------------------------
        # Create directory structure
        # ------------------------------

        # Main directory (named after the key)
        os.makedirs(key, exist_ok=True)

        # Subdirectory named DASE
        dase_dir = os.path.join(key, "DASE")
        os.makedirs(dase_dir,exist_ok=True)

        # File inside the DASE directory
        filename = os.path.join(dase_dir, f"{key}_DASE_genes.txt")

        # Write one gene per line
        with open(filename, "w") as f:
            for gene in sorted(shared_genes):
                f.write(gene + "\n")

        print(f"Written: {filename}")


In [ ]:
from IPython.display import display

In [ ]:
D_merged=D_comparison|D_shared

In [ ]:
for comp in D_merged:
    comp_dir = os.path.join(os.getcwd(), comp,'DASE_Enrichment')
    os.makedirs(comp_dir, exist_ok=True)
    for enrichment in librerie:
        enrich_dir = os.path.join(comp_dir, enrichment)
        gene_list = D_merged[comp]
        #print(EnrichR.get_enrichment(gene_list,library_name=enrichment))
        result = EnrichR.get_enrichment(gene_list, library_name=enrichment)
        # extract the dictionary of results
        enrich_dict = result[0]   
        Library_name = result[1]  

        rows = enrich_dict[Library_name]

        # convert rows into a DataFrame
        df = pd.DataFrame(rows, columns=[
        "Rank",
        "Term name",
        "P-value",
        "Z-score",
        "Combined score",
        "Overlapping genes",
        "Adjusted p-value",
        "Old p-value",
        "Old adjusted p-value"])
        # Add a new column counting the number of genes in 'Overlapping genes'
        df['number of Genes'] = df['Overlapping genes'].apply(lambda x: len(str(x).split(',')))
        df=df[df['Adjusted p-value']<=0.05]
        if df is not None and not df.empty:
            print(f"Resulting enrichment for {comp} with {enrichment} library")
            print(f"Number of genes: {len(gene_list)}")
            print(df[['Term name',"Adjusted p-value",'number of Genes']])
            # save DataFrame as CSV
            csv_path = os.path.join(enrich_dir, f"{comp}_{enrichment}.csv")
            df.to_csv(csv_path, index=False)
            
            # save the plot
            plot_path = os.path.join(enrich_dir, f"{comp}_{enrichment}.png")
            plot_combined_score(df, save_path=plot_path, dpi=300)
